# Bronze — dados do dashboard Superadmin
Conecta ao PostgreSQL e salva snapshots Delta de chamados de suporte, usuários e controle do RPA. Execute em compute com suporte a Python/PySpark e selecione o catálogo/schema de destino no Databricks. Cada execução substitui os snapshots Bronze anteriores.

In [ ]:
SECRET_SCOPE = "skadi-bi"

JDBC_URL = dbutils.secrets.get(scope=SECRET_SCOPE, key="postgres-jdbc-url")
JDBC_USUARIO = dbutils.secrets.get(scope=SECRET_SCOPE, key="postgres-usuario")
JDBC_SENHA = dbutils.secrets.get(scope=SECRET_SCOPE, key="postgres-senha")

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

# Uma consulta constante confirma a conexão sem consultar tabelas do sistema.
conexao = (
    spark.read.format("jdbc")
    .option("url", JDBC_URL)
    .option("user", JDBC_USUARIO)
    .option("password", JDBC_SENHA)
    .option("driver", "org.postgresql.Driver")
    .option("query", "SELECT 1 AS conexao_ok")
    .load()
)
conexao.collect()
print("Conexão com o PostgreSQL validada.")

## Fontes e indicadores
- Chamados: uma linha por solicitação de suporte. As datas de abertura e encerramento permitem filtros e evolução por período; status, categoria e responsável permitem agrupamentos. As duas datas permitem calcular o tempo de resolução posteriormente.
- Usuários: uma linha por usuário, com nome para identificar responsáveis e nível de acesso para contagens. Sem CPF, senha, username ou e-mail. O total é um retrato atual: a fonte não possui data de cadastro para evolução histórica.
- RPA: uma linha por carga configurada, com a última execução bem-sucedida. Não representa tentativas ou falhas.
Neste dashboard, atendimentos por status corresponde aos chamados de suporte: REGISTRADO, EM_ATENDIMENTO e ATENDIDO. A tabela de atendimento de alertas não faz parte deste escopo. O Bronze mantém os valores da fonte; regras e KPIs serão calculados nas camadas seguintes.

In [ ]:
from pyspark.sql import functions as F

def ler_postgres(consulta):
    return (
        spark.read.format("jdbc")
        .option("url", JDBC_URL)
        .option("user", JDBC_USUARIO)
        .option("password", JDBC_SENHA)
        .option("driver", "org.postgresql.Driver")
        .option("fetchsize", "10000")
        .option("query", consulta)
        .load()
    )

FONTES = {
    "bronze_site_chamados": """
        SELECT id, cod_usuario_solicitante, categoria, status,
               data_hora_abertura, data_hora_encerramento,
               cod_usuario_responsavel
        FROM public.tb_solicitacao_suporte
    """,
    "bronze_site_usuarios": """
        SELECT id, nome, nivel_acesso
        FROM public.tb_usuario
    """,
    "bronze_site_controle_rpa": """
        SELECT nome_carga, ultima_execucao_sucesso
        FROM public.tb_controle_rpa
    """,
}

for tabela, consulta in FONTES.items():
    dados = ler_postgres(consulta).withColumn("ingestado_em", F.current_timestamp())
    (
        dados.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )
    print(f"Snapshot salvo: {tabela}")

print("Ingestão Bronze concluída.")